In [2]:
import os

# 1. 设定知识库文档所在的文件夹路径
docs_dir = 'D:/coffee-rag/knowledge_base'

# 2. 初始化 chunks 列表，用于存放所有切分好的块
chunks = []    # 每个元素四样信息：文档名、来源、小节标题、正文

# 3. 遍历文件夹下的所有文件
for fname in sorted(os.listdir(docs_dir)): #os.listdir()获取所有文件的名字，sorted()按顺序返回
    if not fname.endswith('.md'):  # 跳过非 md 文件
        continue

    # 4. 构建完整的文件路径并读取内容
    file_path = os.path.join(docs_dir, fname) #形成完整的绝对路径
    text = open(file_path, encoding='utf-8').read()

    # 5. 提取来源：逐行找 "- 来源: " 开头的那一行
    source = ''
    for line in text.splitlines(): #按换行符切分
        if line.startswith('- 来源：  '):
            source = line
            break

    # 6. 按 '## ' 切块（跳过第0段标题前的内容）
    sections = text.split('## ') #会在列表第0位留下 第一个 ## 之前的内容
    for sec in sections[1:]:          # 第 0 段是标题前的内容，丢弃
        title = sec.split('\n', 1)[0]        # 小节标题（第一行）。对每个小节按换行符切分一次。1表示只切一次
        body  = sec.split('\n', 1)[1].strip() # 正文（剩下部分）。.strip()去除正文首尾多余的空白字符。

      # 7. 将提取出的信息打包成字典，追加到 chunks 列表
        chunks.append({
            "doc_name": fname,
            "source": source,
            "title": title,
            "body": body
        })

    

# ================= 检查点验证 =================
# 打印总块数（预期 30 上下）
print(f"总块数: {len(chunks)}")

# 打印第一块
if chunks:
    print("\n第一块内容:")
    print(chunks[0])

# 打印每篇文档切出的块数（用熟悉的 len() 思路）
print("\n每篇文档切出的块数:")
doc_counts = {}
for chunk in chunks:
    doc_name = chunk["doc_name"]
    doc_counts[doc_name] = doc_counts.get(doc_name, 0) + 1

for doc, count in doc_counts.items():
    print(f"{doc}: {count} 块")
  

总块数: 33

第一块内容:
{'doc_name': '01_2023官方口径_政协答复.md', 'source': '- 来源：https://nync.yn.gov.cn/html/2024/jytabl2024_0611/1416438.html', 'title': '关键事实', 'body': '1. 2023年全省咖啡种植面积114.6万亩，产量14.6万吨，均占全国的98%以上。口径为云南省农业农村厅答复中引用的“统计数据”（2023年自然年）。云南咖啡产业已逐步形成从品种研究、种植、初加工、精深加工、交易集散、装备制造、产业服务的全产业链体系。\n\n2. 出口数据（口径：昆明海关）：2023年云南省咖啡出口9857.44吨，同比下降68.48%；出口金额3.66亿元，同比下降62.09%。出口结构为：生咖啡豆出口6950.17吨，占70.5%；焙炒咖啡出口141吨，占1.43%；浓缩精汁出口530.2吨，占5.38%；咖啡制品出口2236.07吨，占22.68%（占比为占全省咖啡出口总量的比例）。\n\n3. 资源禀赋与产业定位：云南拥有热区面积8.11万平方公里；咖啡产业是云南高原特色现代农业“1+10+3”重点产业之一；发展基本思路为以精品咖啡为引领，着力提升咖啡精品率和精深加工率，大力推动一二三产业融合。\n\n4. 政策与资金（2022年以来）：省农业农村厅先后印发《关于支持咖啡产业精品率、精深加工率提升的若干政策措施》《云南省精品咖啡种植区划》《云南省精品咖啡庄园认定管理办法（试行）》等文件；并争取到1.5亿元中央财政资金支持云南咖啡产业集群项目，用于打造精品咖啡种植基地、推进鲜果集中处理中心建设、新建或提升改造咖啡精深加工生产线、加强产业宣传推广等。\n\n5. 补助政策：2022年8月2日，省农业农村厅、省财政厅、省工业和信息化厅联合印发《关于推动咖啡精品率和精深加工率提升若干政策措施的通知》（云农绿〔2022〕8号），提出绿色发展、鲜果集中采处、精深加工、品牌打造、建设精品庄园、加大金融支持6个方面重点工作与支持政策；2022年该项政策共奖补绿色发展资金481.48万元。对获得绿色食品、有机产品认证的产品及基地实行“先建后补、获证后补、按量奖补”一次性补贴；对获得国家地理标志证明商标注册和地理标志产品保护且授权用标的咖啡产品

In [4]:
#修改版本的切块代码
import os

# 1. 设定知识库文档所在的文件夹路径
docs_dir = 'D:/coffee-rag/knowledge_base'
chunks = []

# 2. 遍历文件夹下的所有文件
for fname in sorted(os.listdir(docs_dir)):
    if not fname.endswith('.md'):
        continue

    file_path = os.path.join(docs_dir, fname)
    text = open(file_path, encoding='utf-8').read()

    # 3. 提取文档级来源：放宽匹配，以防原格式有变
    doc_source = ''     #这个 doc_source 是兜底来源：如果某个块的标题里没有单独写 （来源：...），就用这个文档级来源。
    lines = text.splitlines()
    for i,line in enumerate(lines):
        if line.startswith('- 来源：'):
            if line.strip() == '- 来源：':  #如果是裸行，往下收集缩进的行
                more = []  # 先把缩进行收集进列表，最后用分号拼接（比空格好读）
                for j in range(i + 1 , len(lines)):
                    next_line = lines[j]
                    # 以空格或 tab 开头，且非空行，就是缩进的 URL
                    if (next_line.startswith(' ') or next_line.startswith('\t')) and next_line.strip():
                        more.append(next_line.strip().lstrip('- '))  # 去掉开头的 '- ' 子弹头
                    else:
                        break  # 遇到非缩进行（如 - 发布日期：）停止收集
                doc_source = '- 来源：' + '；'.join(more)
            else:
                  # 如果本身就带有 URL，直接取
                doc_source = line.strip()
            break

    # 4. 【修改点1：显式切分】
    # 用 '\n## ' 切小节，避免误切 '### '
    sections = text.split('\n## ')
    
    for sec in sections[1:]:  # 第0段是文档大标题及元数据，丢弃。第一个##之前的内容。
        # 提取当前 ## 层的标题和正文
        lines = sec.split('\n', 1)   #只切一次换行
        title = lines[0].strip()  #取标题，去掉首尾空白
        body = lines[1] if len(lines) > 1 else ''   #如果存在正文就取，否则给空字符串，防止后面报错。

        # 检查当前小节内部是否有 '### ' 三级标题
        sub_sections = body.split('\n### ')
        '''如果正文里没有三级标题，sub_sections 长度为 1，sub_sections[0] 就是整个正文。
        如果正文里有三级标题，sub_sections[0] 是二级标题下、第一个三级标题之前的内容；后面的每个元素就是三级标题块'''
       
        # --- 处理 ## 当前块 ---
        body_main = sub_sections[0]   #二级标题下、第一个三级标题之前的正文。
        # 【修改点2：过滤垃圾块】去掉空白后不足20字符的跳过
        if len(body_main.strip()) >= 20:
            # 【修改点3：来源按块提取】
            source = doc_source  # 默认退回文档级
            #1. 优先匹配带有（来源：）的标题
            if '（来源：' in title:
                source = title.split('（来源：', 1)[1]
                if source.endswith('）'):
                    source = source[:-1]  # 剥掉结尾的 '）'
            # 2. 匹配 07 的 "来源1｜xxx" 格式（注意：是全角竖线 ｜，不是半角 |）
            elif title.startswith('来源') and '｜' in title:
                source = title.split('｜', 1)[1].strip()
            # 3. 匹配 07/08 的 "补充来源xxx" 格式
            elif title.startswith('补充来源'):
                source = title
                
            chunks.append({
                "doc_name": fname,
                "source": source,
                "title": title,
                "body": body_main.strip()
            })
            

        # --- 处理 ### 子块（如果有） ---
        for sub_sec in sub_sections[1:]:
            sub_lines = sub_sec.split('\n', 1)
            sub_title = sub_lines[0].strip()
            sub_body = sub_lines[1] if len(sub_lines) > 1 else ''

            # 过滤垃圾块
            if len(sub_body.strip()) >= 20:
                # 来源按块提取：和上面 ## 层用同样的三条规则（修改点4：子块也要识别 来源N｜ 和 补充来源）
                source = doc_source
                if '（来源：' in sub_title:
                    source = sub_title.split('（来源：', 1)[1]
                    if source.endswith('）'):
                        source = source[:-1]
                # 匹配 "来源1｜xxx" 格式（全角竖线）
                elif sub_title.startswith('来源') and '｜' in sub_title:
                    source = sub_title.split('｜', 1)[1].strip()
                # 匹配 "补充来源xxx" 格式
                elif sub_title.startswith('补充来源'):
                    source = sub_title
                
                chunks.append({
                    "doc_name": fname,
                    "source": source,
                    "title": sub_title,
                    "body": sub_body.strip()
                })

# ================= 检查点验证 =================
print(f"总块数: {len(chunks)}")

# 定义一个专门打印某文档 title 和 source 的函数
def print_doc_details(doc_name):
    print(f"\n===== {doc_name} 的明细 =====")
    for chunk in chunks:
        if chunk["doc_name"] == doc_name:
            print(f"title: {chunk['title']}")
            print(f"source: {chunk['source']}")
            print("-" * 30)

# ① 打印 04 的 title 和 source
print_doc_details("04_出口数据_海关.md")

# ② 打印 07、08 的 title 和 source
print_doc_details("07_贵阳咖啡之城.md")
print_doc_details("08_贵阳咖博会_新华社.md")

# ③ 打印每篇文档切出的块数
print("\n每篇文档切出的块数:")
doc_counts = {}
for chunk in chunks:
    doc_name = chunk["doc_name"]
    doc_counts[doc_name] = doc_counts.get(doc_name, 0) + 1

for doc, count in doc_counts.items():
    print(f"{doc}: {count} 块")

总块数: 30

===== 04_出口数据_海关.md 的明细 =====
title: 一、2026年上半年云南咖啡出口（来源：昆明海关网站文章，同稿经新浪、新黄河/七彩云端转载页核对）
source: 昆明海关网站文章，同稿经新浪、新黄河/七彩云端转载页核对
------------------------------
title: 二、2026年前8个月云南外贸与咖啡出口（来源：云南网）
source: 云南网
------------------------------
title: 三、2026年一季度普洱咖啡出口（来源：搜狐·开屏新闻）
source: 搜狐·开屏新闻
------------------------------
title: 口径与备注
source: - 来源：昆明海关网站《上半年云南咖啡出口量值领跑全国》：http://sjs.customs.gov.cn/customs/2026-07/27/article_2026072715532616892.html；云南网《前8个月云南省进出口1946.7亿元 增长8.3%》：http://yn.yunnan.cn/system/2026/09/30/034164022.shtml；搜狐·开屏新闻《2026年一季度，普洱咖啡出口1.1亿元、远销19个国家和地区》：https://www.sohu.com/a/1006934915_100260304；同稿核对页（昆明海关网站文章因反爬无法直取，经以下两个独立转载页核实同稿全文）：https://news.sina.com.cn/zx/gj/2026-07-25/doc-iniizkxf1081942.shtml ；https://www.jinantimes.com.cn/news-107-10817207.html
------------------------------

===== 07_贵阳咖啡之城.md 的明细 =====
title: 来源1｜贵阳市人民政府门户网站，2026-09-29（原载贵阳日报）
source: 贵阳市人民政府门户网站，2026-09-29（原载贵阳日报）
------------------------------
title: 来源2｜贵阳市人民政府门户网站，2026-09-28（原载贵阳日报）

In [9]:
import math
import jieba
from collections import Counter

# 1. 分词函数
def tokenize(text):
    # jieba.lcut 精确模式分词，并过滤掉纯空白字符
    return [word for word in jieba.lcut(text) if word.strip()]

# 2. 建索引函数
def build_index(chunks):
    N = len(chunks)
    df = {}  #记录每个词出现在多少个文档中
    tokenized_bodies = [] #缓存所有文档的分词结果，避免后续重复分词
    vocab = set() #词表，记录所有出现过的唯一词

    # 第一步：遍历所有块，统计词频和文档频率 (DF)
    for chunk in chunks:
        tokens = tokenize(chunk['body'])
        tokenized_bodies.append(tokens)
        
        # 对当前块的词去重，用于统计 DF
        unique_tokens = set(tokens)  #set(tokens) 去重，因为 DF 统计的是“出现该词的文档数”，同一个文档内出现多次只算一次
        for word in unique_tokens:
            df[word] = df.get(word, 0) + 1
            vocab.add(word)

    # 第二步：计算 IDF
    # 公式：idf(w) = log((N+1) / (df(w)+1)) + 1。加 1 是为了防止 df 为 0 或者 df=N 时导致 log 结果为 0 或负数，保持 IDF 的基本权重
    idf = {}
    for word in vocab:
        idf[word] = math.log((N + 1) / (df[word] + 1)) + 1

    # 第三步：构建每个块的 TF-IDF 向量
    doc_vectors = []
    for tokens in tokenized_bodies:
        tf = Counter(tokens)  #使用 Counter 统计每个块中每个词的出现次数     
        doc_len = len(tokens)
        # 公式：tf(w,d) = 词w在块d出现次数 / 块d总词数
        # 向量：tf * idf
        vec = {word: (tf[word] / doc_len) * idf[word] for word in tf}
        doc_vectors.append(vec)

    return doc_vectors, idf, N

# 3. 检索函数
def search(question, top_k=5):
    # 查询分词
    q_tokens = tokenize(question)
    q_tf = Counter(q_tokens)
    q_len = len(q_tokens)
    
    # 构建查询向量（同样使用 TF-IDF）
    # 注意：如果查询词不在词表中，IDF 默认为 log((N+1)/1)+1，以兼容未登录词
    q_vec = {}
    for word in q_tf:
        #只保留存在于 IDF 词表中的词（丢弃未登录词）
        if word in idf:
            q_vec[word] = (q_tf[word] / q_len) * idf[word]

     # 注意：如果 query 的所有词都不在词表里，q_vec 为空，模长为 0
    # 所以 q_norm 计算时需要防零
    
    # 计算查询向量的模长
    q_norm = math.sqrt(sum(v ** 2 for v in q_vec.values()))
    if q_norm == 0:
        print(f"查询: '{question}' -> 没有任何有效词汇能匹配到知识库，直接拒答。")
        return
    
    # 遍历所有文档，计算余弦相似度
    results = []
    for i, doc_vec in enumerate(doc_vectors):
        # 计算点积
        dot_product = sum(q_vec.get(w, 0) * doc_vec.get(w, 0) for w in q_vec)
        
        # 计算文档向量的模长
        doc_norm = math.sqrt(sum(v ** 2 for v in doc_vec.values()))
        
        # 余弦相似度，分母加 1e-8 防止除以零
        sim = dot_product / (q_norm * doc_norm + 1e-8)
        results.append((sim, i))

    # 按相似度降序排序，取 Top-K
    results.sort(key=lambda x: x[0], reverse=True)
    top_results = results[:top_k]

    # 打印格式化结果
    print(f"查询: '{question}'")
    for rank, (sim, idx) in enumerate(top_results, 1):
        chunk = chunks[idx]
        body_preview = chunk['body'][:80].replace('\n', ' ') + "..." if len(chunk['body']) > 80 else chunk['body']
        print(f"Top {rank} | 相似度: {sim:.4f} | 文档: {chunk['doc_name']} | 标题: {chunk['title']}")
        print(f"        预览: {body_preview}")
    print("-" * 50)

# 4. 初始化索引并运行检查点
doc_vectors, idf, N = build_index(chunks)

# 检查点 1
search('86.6亿', 3)
# 检查点 2
search('贵阳咖啡门店数量', 3)
# 检查点 3
search('2025年云南咖啡产量', 3)

查询: '86.6亿'
Top 1 | 相似度: 0.0771 | 文档: 02b_2025发布会初步口径_1月8日.md | 标题: 口径与备注
        预览: - 本文档 146.3 万亩/13.89 万吨为**初步口径**，2026-09-17 热作大会修正为 151.88 万亩/17.07 万吨（见 02 号文档）...
Top 2 | 相似度: 0.0564 | 文档: 02b_2025发布会初步口径_1月8日.md | 标题: 关键事实
        预览: 1. 2025 年云南咖啡种植面积 146.3 万亩，同比增长 15.4%（初步口径，后热作大会修正为 151.88 万亩） 2. 2025 年生豆产量 13....
Top 3 | 相似度: 0.0000 | 文档: 01_2023官方口径_政协答复.md | 标题: 关键事实
        预览: 1. 2023年全省咖啡种植面积114.6万亩，产量14.6万吨，均占全国的98%以上。口径为云南省农业农村厅答复中引用的“统计数据”（2023年自然年）。云南...
--------------------------------------------------
查询: '贵阳咖啡门店数量'
Top 1 | 相似度: 0.2587 | 文档: 07_贵阳咖啡之城.md | 标题: 来源4｜腾讯新闻·贵州省文化和旅游厅官方账号，2026-09-12
        预览: https://news.qq.com/rain/a/20260912A044J200  1. 规模与本土品牌：截至2025年末，贵阳全市咖啡门店突破3000家...
Top 2 | 相似度: 0.2134 | 文档: 08_贵阳咖博会_新华社.md | 标题: 补充来源：贵州广电系媒体现场报道
        预览: 补充来源1：动静新闻（贵州广播电视台）《不产咖啡的贵阳，如何撑起一场国际咖啡产业盛会？》（记者李柏杉、王琰，2026-09-29）：https://moveme...
Top 3 | 相似度: 0.2019 | 文档: 07_贵阳咖啡之城.md | 标题: 来源1｜贵阳市人民政府门户网站，2026-09-29（原载贵阳日报）
        预览: https://www.guiyang.gov.cn

In [11]:
from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv('D:/coffee-rag/.env')   # 用绝对路径——记住内核 CWD 的教训
client = OpenAI(api_key=os.getenv('DEEPSEEK_API_KEY'), base_url='https://api.deepseek.com')

SYSTEM_PROMPT ="""
# 角色设定
你是一个严谨的“云南与贵阳咖啡产业”知识库问答助手。你的唯一任务是基于系统提供的参考资料，准确、客观地回答用户关于云南与贵阳咖啡产业的问题。

# 核心规则（四道反幻觉防线，请严格遵守）
1. **绝对忠实于资料**：你只能使用检索注入的参考资料来回答问题。绝对禁止使用你训练记忆中关于云南与贵阳咖啡的任何数字、事实或判断。
2. **强制标来源**：你回答中的每一个数据、事实或结论，都必须注明出处。请在相关语句末尾标注对应的资料编号，格式如 `[资料1]`、`[资料2]`。
3. **拒绝编造**：如果参考资料里找不到回答用户问题所需的信息，你必须如实回答“资料中没有相关信息”，严禁推测、编造或拼凑答案。
4. **口径差异必须说明**：如果同一项数据在资料中出现了不同的统计口径（例如：农业产值 86.6 亿元与全产业链产值 995 亿元），你必须将不同口径的数据全部列出，并解释它们统计范围的区别，绝对不允许只挑其中一个口径回答。
5. **数字纪律**：引用数字必须与资料原文逐字一致，不得四舍五入、改写单位，不得自行添加或删除“约”“达”“超”等修饰词。资料写“约17万吨”就写“约17万吨”，资料写“17.07万吨”就写“17.07万吨”，不可混淆。

# 回答风格
保持专业、客观、简洁。回答时优先陈述事实，如果资料间的数据存在冲突或口径不同，需按规则4进行处理。"""

def search(question, top_k=5):
    q_tokens = tokenize(question)
    q_tf = Counter(q_tokens)
    q_len = len(q_tokens)
    
    q_vec = {}
    for word in q_tf:
        if word in idf: # 过滤未登录词，修复 Q1 分数稀释 Bug
            q_vec[word] = (q_tf[word] / q_len) * idf[word]
        
    q_norm = math.sqrt(sum(v ** 2 for v in q_vec.values()))
    if q_norm == 0:
        print(f"查询: '{question}' -> 没有任何有效词汇能匹配到知识库。")
        return []

    results = []
    for i, doc_vec in enumerate(doc_vectors):
        dot_product = sum(q_vec.get(w, 0) * doc_vec.get(w, 0) for w in q_vec)
        doc_norm = math.sqrt(sum(v ** 2 for v in doc_vec.values()))
        sim = dot_product / (q_norm * doc_norm + 1e-8)
        results.append((sim, i))

      
    results.sort(key=lambda x: x[0], reverse=True)
    return results[:top_k]


def build_prompt(question, results):
    prompt_parts = []
    prompt_parts.append("# 检索到的参考资料")
    prompt_parts.append("以下是系统检索到的相关知识块，请严格依据它们回答问题。\n")
    
    for i, (sim, idx) in enumerate(results, 1):
        chunk = chunks[idx]
        prompt_parts.append(f"[资料{i}]")
        prompt_parts.append(f"标题：{chunk['title']}")
        prompt_parts.append(f"来源：{chunk['source']}")
        prompt_parts.append(f"正文：{chunk['body']}\n") # 全文注入，不截断
        
    prompt_parts.append("# 用户提问")
    prompt_parts.append(question)
    
    return "\n".join(prompt_parts)

def ask(question, top_k=5):
    print(f"\n🔍 查询: '{question}'")
    
    # 1. 检索
    top_results = search(question, top_k)
    if not top_results:
        print("❌ 检索不到相关资料，直接拒答。")
        return None
        
    # 2. 打印检索留痕（文档名 | 标题 | 相似度）
    print("--- 检索 Top-3 留痕 ---")
    for rank, (sim, idx) in enumerate(top_results, 1):
        chunk = chunks[idx]
        print(f"Top {rank} | {chunk['doc_name']} | {chunk['title']} | 相似度: {sim:.4f}")
    print("-" * 30)
        
    # 3. 拼接用户消息
    user_prompt = build_prompt(question, top_results)
    
    # 4. 组装 messages
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]
    
    # 5. 调用 API (temperature=0.1 减少幻觉)
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            temperature=0.1
        )
        answer = response.choices[0].message.content
        print("\n🤖 模型回答:\n" + answer)
        return answer
    except Exception as e:
        print(f"\n❌ API 调用失败: {e}")
        return None

# ================= 6. 执行检查点 =================
print("="*60)
print("【题1：事实问】")
ask('2025年云南省咖啡种植面积多少？')

print("\n" + "="*60)
print("【题2：口径陷阱问】")
ask('2025年云南咖啡出口了多少万吨？')

print("\n" + "="*60)
print("【题3：库外问】")
ask('贵阳有多少家星巴克？')

【题1：事实问】

🔍 查询: '2025年云南省咖啡种植面积多少？'
--- 检索 Top-3 留痕 ---
Top 1 | 01_2023官方口径_政协答复.md | 关键事实 | 相似度: 0.1663
Top 2 | 02b_2025发布会初步口径_1月8日.md | 关键事实 | 相似度: 0.1631
Top 3 | 03_2024产业报告.md | 关键事实 | 相似度: 0.1496
------------------------------

🤖 模型回答:
根据资料2，2025年云南咖啡种植面积有两个口径的数据：

- **初步口径**：146.3万亩，同比增长15.4%[资料2]
- **修正口径**：后经热作大会修正为151.88万亩[资料2]

两个数据均出自同一资料，其中146.3万亩为发布会初步口径，151.88万亩为后续热作大会修正口径[资料2]。

【题2：口径陷阱问】

🔍 查询: '2025年云南咖啡出口了多少万吨？'
--- 检索 Top-3 留痕 ---
Top 1 | 06_农业农村部月度动态.md | 关键事实 | 相似度: 0.1582
Top 2 | 04_出口数据_海关.md | 二、2026年前8个月云南外贸与咖啡出口（来源：云南网） | 相似度: 0.1545
Top 3 | 02b_2025发布会初步口径_1月8日.md | 关键事实 | 相似度: 0.1495
------------------------------

🤖 模型回答:
资料中没有直接给出“2025年云南咖啡出口了多少万吨”这一数据。

资料中与2025年云南咖啡出口相关的信息如下：

- 2025年3.25万吨优质“云咖”远销荷兰、德国、美国等29个国家和地区，出口量同比增长358%（省厅发布会口径）[资料3]。
- 该口径与昆明海关“咖啡豆及制品1.9万吨、43国”的品类口径不同[资料3]。

因此，若按省厅发布会口径，2025年云南咖啡出口量为3.25万吨；但资料同时提示存在昆明海关口径（咖啡豆及制品1.9万吨），两者统计范围不同，不能混用[资料3]。

【题3：库外问】

🔍 查询: '贵阳有多少家星巴克？'
--- 检索 Top-3 留痕 ---
Top 1 | 08_贵阳咖博会_新华社.md | 补充来源：贵州广电系媒体现场

'资料中没有相关信息。\n\n检索到的参考资料中提到了贵阳咖啡经营主体超3000家、本土品牌占比接近80%[资料1]，以及注册咖啡馆数量超过3000家、密度约2000人/家[资料3]，但均未提及星巴克在贵阳的具体门店数量。'